# Assignment 1.3: Seq2Seq Summarization with LSTM

**Objective:** Build an encoder-decoder LSTM model for abstractive text summarization.

### Steps:
1. Prepare a small dataset (144 news articles + summaries)
2. Preprocess the text (tokenize, pad)
3. Build an Encoder-Decoder LSTM model
4. Train the model
5. Evaluate and generate summaries

In [ ]:
!pip install tensorflow numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

TensorFlow version: 2.21.0
All libraries imported successfully!


## Step 2: Dataset — 144 News Article + Summary Pairs

In [ ]:
raw_data = [
    ("Scientists discovered a new species of fish in the deep ocean that produces its own light using bioluminescence.", "New glowing fish species found in deep ocean."),
    ("The government announced a new policy to reduce carbon emissions by 50 percent over the next decade.", "Government targets 50 percent emission cut in ten years."),
    ("A major earthquake struck the coastal region causing widespread damage and prompting emergency evacuations.", "Earthquake hits coast causing damage and evacuations."),
    ("Researchers at the university developed a new vaccine that shows 95 percent effectiveness against the flu virus.", "New flu vaccine proves 95 percent effective."),
    ("The stock market reached an all-time high today driven by strong earnings reports from technology companies.", "Stock market hits record high on tech earnings."),
    ("A local farmer discovered ancient coins buried in his field dating back over two thousand years.", "Ancient coins unearthed on local farm."),
    ("The city council approved a plan to build a new public library in the downtown area next year.", "Council approves downtown library construction."),
    ("Electric vehicle sales surged this year as more consumers switch from gasoline powered cars to cleaner alternatives.", "Electric vehicle sales surge as consumers switch."),
    ("A team of explorers reached the summit of the world highest mountain despite harsh weather conditions.", "Explorers summit world highest mountain."),
    ("Scientists detected signals from a distant galaxy that may provide clues about the origin of the universe.", "Scientists detect signals from distant galaxy."),
    ("The annual music festival attracted over one hundred thousand visitors from across the country.", "Music festival draws massive crowd nationwide."),
    ("Doctors found that regular exercise significantly reduces the risk of heart disease in adults over fifty.", "Exercise lowers heart disease risk in older adults."),
    ("A new bridge connecting the two cities was inaugurated today after three years of construction.", "New bridge links two cities after three years."),
    ("The company announced plans to expand its operations to five new countries in the coming year.", "Company plans expansion to five new countries."),
    ("Heavy rainfall caused flooding in low lying areas forcing hundreds of families to leave their homes.", "Flooding displaces hundreds of families."),
    ("A new smartphone model was released featuring an improved camera and longer battery life.", "New smartphone launched with better camera and battery."),
    ("The president signed a new trade agreement that aims to boost exports and create thousands of jobs.", "President signs trade deal to create jobs."),
    ("Animal conservation efforts led to a significant increase in the population of endangered tigers.", "Tiger population rises due to conservation efforts."),
    ("Students at the local school won first place in the national science competition for their innovative project.", "Local students win national science competition."),
    ("Astronomers captured the first ever clear image of a black hole located millions of light years away.", "Astronomers photograph black hole millions of light years away."),
    ("The new hospital wing was opened providing additional beds and advanced medical equipment for patients.", "New hospital wing opens with advanced equipment."),
    ("Climate scientists warn that rising sea levels could threaten coastal cities within the next fifty years.", "Rising seas threaten coastal cities scientists warn."),
    ("A renowned artist unveiled a massive sculpture in the city center attracting thousands of visitors daily.", "Giant sculpture unveiled in city center."),
    ("The national football team won the championship after a thrilling match that ended in extra time.", "National team wins championship in extra time."),
    ("Researchers found that eating a Mediterranean diet can improve mental health and reduce depression.", "Mediterranean diet linked to better mental health."),
    ("A startup launched an app that uses artificial intelligence to help users manage their daily tasks efficiently.", "AI app launched to help manage daily tasks."),
    ("The volcano erupted overnight sending ash clouds over nearby towns and disrupting air travel.", "Volcano erupts disrupting towns and air travel."),
    ("Scientists completed the full mapping of the human genome which could revolutionize personalized medicine.", "Full human genome mapped for personalized medicine."),
    ("The charity organization raised five million dollars to provide clean water to rural communities.", "Charity raises millions for rural clean water access."),
    ("A new law requires all restaurants to display calorie counts on their menus to promote healthy eating.", "New law mandates calorie counts on restaurant menus."),
    ("Engineers developed a solar powered drone capable of flying continuously for thirty days without landing.", "Solar drone can fly for thirty days non stop."),
    ("The film won six awards at the international festival including best picture and best director.", "Film wins six awards at international festival."),
    ("Health officials reported a decline in childhood obesity rates following a nationwide awareness campaign.", "Childhood obesity drops after awareness campaign."),
    ("A historic peace agreement was signed between two neighboring countries ending decades of conflict.", "Peace deal ends decades of conflict between neighbors."),
    ("The technology giant launched a cloud computing service promising faster speeds and lower costs for businesses.", "Tech giant launches faster cheaper cloud service."),
    ("Archaeologists uncovered an ancient city beneath the desert sands revealing advanced early civilization.", "Ancient city discovered beneath desert sands."),
    ("A breakthrough in battery technology could allow electric cars to travel over a thousand kilometers on a single charge.", "Battery breakthrough boosts electric car range."),
    ("The central bank raised interest rates to combat rising inflation affecting household budgets.", "Central bank hikes rates to fight inflation."),
    ("Scientists developed a biodegradable plastic that breaks down in weeks instead of hundreds of years.", "New plastic breaks down in weeks not centuries."),
    ("The marathon runner set a new world record completing the race in under two hours for the first time in history.", "Runner sets world record completing marathon under two hours."),
    ("A wildfire spread rapidly across dry woodland destroying hundreds of homes and forcing evacuations.", "Wildfire destroys homes and forces evacuations."),
    ("Doctors performed the first successful heart transplant using an organ grown in a laboratory.", "First lab grown heart transplant performed successfully."),
    ("The government launched a free coding education program for unemployed youth to boost digital skills.", "Government offers free coding classes for unemployed youth."),
    ("A rare blue diamond was sold at auction for a record breaking price of fifty million dollars.", "Rare blue diamond fetches record fifty million at auction."),
    ("Scientists found microplastics in human blood for the first time raising serious health concerns.", "Microplastics detected in human blood for first time."),
    ("The country achieved universal broadband internet coverage connecting even the most remote villages.", "Country reaches universal broadband internet coverage."),
    ("A new study found that sleep deprivation significantly impairs decision making and memory retention.", "Study links sleep loss to poor decisions and memory."),
    ("The airline announced the launch of the first direct flight route between two major continents.", "Airline launches first direct intercontinental route."),
    ("Farmers are adopting drone technology to monitor crops and apply fertilizers more precisely.", "Drones help farmers monitor crops more efficiently."),
    ("A new app uses facial recognition to help identify missing persons and reunite them with families.", "Facial recognition app helps find missing persons."),
    ("The museum opened a new wing dedicated to ancient Egyptian artifacts including mummies and hieroglyphics.", "Museum opens wing for Egyptian artifacts."),
    ("Researchers created a robot that can perform delicate surgery with greater precision than human surgeons.", "Robot performs surgery with superhuman precision."),
    ("The city introduced bike sharing lanes and stations to reduce car traffic in the urban center.", "City adds bike lanes to cut urban traffic."),
    ("Scientists confirmed that a nearby asteroid poses no threat to Earth for the next century.", "Asteroid confirmed safe from Earth for hundred years."),
    ("A heatwave broke temperature records across the region with temperatures exceeding forty five degrees.", "Record heatwave hits region with extreme temperatures."),
    ("The bestselling author released a new novel that sold one million copies in the first week.", "New novel sells one million copies in first week."),
    ("Renewable energy sources now account for over forty percent of the country total electricity generation.", "Renewables supply forty percent of national electricity."),
    ("A community initiative planted one million trees across urban areas to combat air pollution.", "Million trees planted to fight urban air pollution."),
    ("Scientists discovered that dolphins communicate using complex vocal patterns similar to human language.", "Dolphins use complex vocal patterns like human speech."),
    ("The university announced full scholarships for students from low income families starting next year.", "University offers full scholarships for low income students."),
    ("A new blood test can detect cancer up to four years before symptoms appear improving survival rates.", "Blood test detects cancer four years early."),
    ("The fashion brand committed to using only sustainable materials in all its products by next decade.", "Fashion brand pledges fully sustainable materials."),
    ("An international team of scientists launched a mission to study the effects of climate change on Arctic ice.", "Scientists launch Arctic ice climate study mission."),
    ("The government invested heavily in high speed rail to connect major cities and reduce highway congestion.", "Government invests in high speed rail for cities."),
    ("Researchers found evidence of liquid water beneath the surface of Mars raising hopes for life.", "Liquid water found under Mars surface."),
    ("A new education app personalizes learning for students using artificial intelligence algorithms.", "AI app personalizes learning for each student."),
    ("The factory switched to renewable energy cutting its carbon footprint by seventy percent.", "Factory cuts emissions seventy percent with renewables."),
    ("A famous painting stolen from the museum twenty years ago was finally recovered by police.", "Stolen painting recovered after twenty years."),
    ("Health workers vaccinated over a million people in one month in response to a disease outbreak.", "Million people vaccinated in disease outbreak response."),
    ("The tech firm unveiled a new chip that processes data ten times faster than current models.", "New chip processes data ten times faster."),
    ("An expedition team returned from Antarctica with ice core samples dating back eight hundred thousand years.", "Antarctica ice cores reveal eight hundred thousand year history."),
    ("Local volunteers cleaned up fifty tons of plastic waste from rivers and beaches over the weekend.", "Volunteers remove fifty tons of plastic from waterways."),
    ("The prime minister announced measures to address the rising cost of living affecting millions of citizens.", "PM announces steps to tackle rising living costs."),
    ("Scientists engineered bacteria that can absorb and break down oil spills in oceans.", "Engineered bacteria break down ocean oil spills."),
    ("A solar farm covering an area of fifty square kilometers was built to power the entire region.", "Massive solar farm built to power entire region."),
    ("The documentary film exposed illegal wildlife trafficking networks operating across several countries.", "Documentary exposes global wildlife trafficking networks."),
    ("Engineers designed a water purification system that can provide clean drinking water using sunlight alone.", "Solar water purifier provides clean drinking water."),
    ("The satellite launched last year has sent back the highest resolution images of Earth ever captured.", "Satellite captures record high resolution Earth images."),
    ("Scientists warned that deforestation is accelerating the loss of biodiversity at an alarming rate.", "Deforestation drives alarming biodiversity loss."),
    ("A pop star announced a world tour spanning forty countries and one hundred concert dates.", "Pop star launches massive forty country world tour."),
    ("The new budget allocates more funds to healthcare education and infrastructure development.", "Budget boosts spending on health education and infrastructure."),
    ("Millions of migrating birds were tracked using tiny GPS devices attached to their wings.", "GPS devices track millions of migrating birds."),
    ("A major cyber attack disrupted banking services across the country affecting millions of customers.", "Cyber attack cripples national banking services."),
    ("The medical team successfully separated conjoined twins in a complex surgery lasting eighteen hours.", "Surgeons separate conjoined twins in eighteen hour operation."),
    ("Authorities seized over a ton of illegal drugs hidden inside shipping containers at the port.", "Police seize ton of drugs from port containers."),
    ("A new study shows that urban green spaces significantly improve residents mental health and wellbeing.", "Green spaces boost mental health in urban areas."),
    ("The space agency revealed plans to send humans to the Moon again within the next five years.", "Space agency plans Moon mission in five years."),
    ("The Olympic games were officially opened with a spectacular ceremony attended by world leaders.", "Olympic games open with spectacular ceremony."),
    ("New research indicates that coffee consumption is linked to a reduced risk of certain cancers.", "Coffee linked to lower cancer risk says new study."),
    ("The social media platform banned thousands of fake accounts spreading political misinformation.", "Platform removes thousands of fake political accounts."),
    ("A new underwater tunnel connecting two coastal cities was completed after six years of construction.", "Underwater tunnel links coastal cities after six years."),
    ("The country reported its lowest unemployment rate in thirty years boosting consumer confidence.", "Unemployment hits thirty year low boosting confidence."),
    ("Scientists discovered a giant underwater mountain range stretching for thousands of kilometers.", "Huge underwater mountain range discovered by scientists."),
    ("The government introduced strict regulations on single use plastics to reduce ocean pollution.", "New rules restrict single use plastics to protect oceans."),
    ("The annual food festival showcased dishes from over fifty countries attracting gourmet lovers worldwide.", "Food festival features cuisine from fifty countries."),
    ("A new gene therapy successfully treated patients with a previously incurable genetic disorder.", "Gene therapy cures previously untreatable genetic disease."),
    ("The city unveiled a smart traffic management system using sensors and AI to ease congestion.", "Smart AI system eases city traffic congestion."),
    ("Solar powered vehicles competed in a race across the continent testing next generation clean transport.", "Solar vehicles race across continent in clean transport test."),
    ("An online learning platform reported a hundred million active users studying various subjects worldwide.", "Online learning platform reaches hundred million users."),
    ("Authorities rescued over two hundred people from a vessel stranded in the middle of the ocean.", "Two hundred rescued from stranded ocean vessel."),
    ("Scientists demonstrated that music therapy helps reduce anxiety and speed recovery in hospital patients.", "Music therapy reduces anxiety and aids hospital recovery."),
    ("A coastal city installed a wave energy system to generate electricity from ocean tides.", "City generates power from ocean wave energy."),
    ("A young inventor created a device that converts air humidity into clean drinking water at low cost.", "Teen invents device to turn air humidity into water."),
    ("The university hospital launched a telemedicine service connecting rural patients with specialist doctors.", "Telemedicine links rural patients with specialists."),
    ("Scientists announced a breakthrough in fusion energy that could provide unlimited clean power.", "Fusion energy breakthrough promises unlimited clean power."),
    ("A record number of tourists visited the national park this year raising concerns about environmental damage.", "Record tourists strain national park environment."),
    ("The government provided free meals to school children from low income families throughout the year.", "Government provides free school meals for poor children."),
    ("Scientists used satellites to track the growth of illegal mining operations in protected forests.", "Satellites expose illegal mining in protected forests."),
    ("A historic cathedral damaged by fire was fully restored after five years of careful renovation.", "Historic cathedral restored five years after fire damage."),
    ("The mobile payment system reached one billion users making cash transactions increasingly rare.", "Mobile payments hit billion users replacing cash."),
    ("Researchers found that spending time in nature for twenty minutes a day reduces stress hormones.", "Twenty minutes in nature daily reduces stress."),
    ("The airline industry reported its strongest recovery since the pandemic with record passenger numbers.", "Airlines record best passenger numbers since pandemic."),
    ("An international summit agreed on emergency measures to protect coral reefs from bleaching.", "Summit agrees on measures to save coral reefs."),
    ("A new telescope detected an exoplanet with conditions that could potentially support life.", "Telescope finds potentially habitable exoplanet."),
    ("The agricultural research institute developed drought resistant crops that thrive with minimal water.", "New crops thrive in drought with minimal water."),
    ("Police used artificial intelligence to predict and prevent crimes in high risk neighborhoods.", "AI helps police predict and prevent crime."),
    ("A humanitarian organization delivered emergency food and medicine to conflict zones.", "Aid group delivers food and medicine to conflict zones."),
    ("The fashion industry is facing pressure to reduce water usage which is extremely high in textile production.", "Fashion industry pressured to cut water use."),
    ("Engineers built a floating solar power station on the lake generating clean electricity for ten thousand homes.", "Floating solar station powers ten thousand homes."),
    ("A rare meteorite containing organic compounds landed in the desert providing clues about early solar system.", "Rare meteorite lands with clues about solar system origins."),
    ("The national park service introduced limits on visitor numbers to protect fragile ecosystems.", "National park limits visitors to protect ecosystem."),
    ("Scientists found that air pollution causes significant cognitive decline in older adults.", "Air pollution linked to cognitive decline in elderly."),
    ("The bicycle manufacturer launched an affordable electric bike aimed at commuters in urban areas.", "Affordable electric bike targets urban commuters."),
    ("A new policy requires companies to report their gender pay gap data publicly every year.", "Companies must publicly report gender pay gap yearly."),
    ("Volunteers created a free app to connect food donors with local charities to reduce food waste.", "Free app links food donors with charities to cut waste."),
    ("A marine biologist discovered a coral reef nearly untouched by human activity in a remote ocean region.", "Pristine coral reef found in remote ocean region."),
    ("The global health organization declared the outbreak contained after rapid vaccination campaigns.", "Outbreak declared contained after swift vaccination drive."),
    ("Scientists designed artificial leaves that can produce clean fuel from sunlight and water.", "Artificial leaves produce clean fuel from sun and water."),
    ("A startup developed biodegradable packaging made from seaweed to replace single use plastics.", "Seaweed packaging offers plastic free alternative."),
    ("The tech conference drew thousands of entrepreneurs and investors from around the world.", "Tech conference attracts global entrepreneurs and investors."),
    ("Scientists confirmed that regular meditation physically changes brain structure improving focus.", "Meditation reshapes brain and improves focus."),
    ("The railway company introduced hydrogen powered trains eliminating diesel emissions on key routes.", "Hydrogen trains replace diesel on key rail routes."),
    ("A major investment fund pledged to divest from fossil fuels by the end of the decade.", "Investment fund to quit fossil fuels by decade end."),
    ("A citizen scientist discovered a new comet using a backyard telescope and shared findings online.", "Amateur astronomer discovers new comet from backyard."),
    ("The government offered subsidies to households installing rooftop solar panels for the first time.", "Government subsidizes rooftop solar for households."),
    ("Scientists grew a miniature human brain organoid in a lab to study neurological diseases.", "Mini brain grown in lab to study neurological disease."),
    ("The online retail company reduced packaging waste by switching to fully recyclable materials.", "Retailer switches to fully recyclable packaging."),
    ("A new radar system can detect drones flying near airports preventing dangerous incidents.", "Radar system detects airport drones to prevent incidents."),
    ("Climate activists staged protests in over one hundred cities demanding immediate action on emissions.", "Climate protests erupt across hundred cities worldwide."),
    ("A neuroscientist developed a brain implant that allows paralyzed patients to control computers with thought.", "Brain implant lets paralyzed patients control computers."),
    ("The annual charity marathon raised two million dollars to fund children cancer research.", "Charity marathon raises two million for child cancer research."),
    ("An international ban on commercial whaling was renewed protecting whale populations for another decade.", "Whaling ban renewed protecting whales for ten more years."),
    ("Scientists observed a penguin colony surviving in temperatures far lower than previously recorded.", "Penguins survive record low temperatures scientists find."),
    ("A new type of concrete absorbs carbon dioxide from the atmosphere helping reduce greenhouse gases.", "New concrete absorbs greenhouse gases from the air."),
]

articles  = [p[0] for p in raw_data]
summaries = ["<start> " + p[1] + " <end>" for p in raw_data]

print(f"Total samples : {len(articles)}")
print(f"Sample article: {articles[0]}")
print(f"Sample summary: {summaries[0]}")

Total samples : 144
Sample article: Scientists discovered a new species of fish in the deep ocean that produces its own light using bioluminescence.
Sample summary: <start> New glowing fish species found in deep ocean. <end>


## Step 3: Preprocessing — Tokenize & Pad

In [ ]:
# Hyperparameters
VOCAB_SIZE  = 3000
MAX_ENC_LEN = 30
MAX_DEC_LEN = 12
EMBED_DIM   = 64
LSTM_UNITS  = 128
BATCH_SIZE  = 16
EPOCHS      = 30

# Tokenize articles
enc_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
enc_tokenizer.fit_on_texts(articles)
enc_padded = pad_sequences(enc_tokenizer.texts_to_sequences(articles),
                           maxlen=MAX_ENC_LEN, padding='post', truncating='post')

# Tokenize summaries
dec_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
dec_tokenizer.fit_on_texts(summaries)
dec_padded = pad_sequences(dec_tokenizer.texts_to_sequences(summaries),
                           maxlen=MAX_DEC_LEN, padding='post', truncating='post')

DEC_VOCAB = min(VOCAB_SIZE, len(dec_tokenizer.word_index) + 1)

# Decoder input (all tokens except last) and target (all tokens except first)
decoder_input  = dec_padded[:, :-1]
decoder_target = dec_padded[:, 1:]
decoder_target_onehot = tf.keras.utils.to_categorical(decoder_target, num_classes=DEC_VOCAB)

print("Encoder input shape :", enc_padded.shape)
print("Decoder input shape :", decoder_input.shape)
print("Decoder target shape:", decoder_target_onehot.shape)
print("Decoder vocab size  :", DEC_VOCAB)

Encoder input shape : (144, 30)
Decoder input shape : (144, 11)
Decoder target shape: (144, 11, 683)
Decoder vocab size  : 683


## Step 4: Build the Encoder-Decoder LSTM Model

```
ENCODER                            DECODER
───────                            ───────
Input → Embedding → LSTM  ──→  LSTM → Dense(softmax) → word predictions
                   (h, c)  ↑
                      Initial state passed from encoder
```

In [ ]:
tf.keras.backend.clear_session()

# ── ENCODER ──
encoder_inputs = Input(shape=(MAX_ENC_LEN,), name="enc_inp")
enc_emb        = Embedding(VOCAB_SIZE, EMBED_DIM, name="enc_emb")(encoder_inputs)
_, state_h, state_c = LSTM(LSTM_UNITS, return_state=True, name="enc_lstm")(enc_emb)
encoder_states = [state_h, state_c]

# ── DECODER ──
decoder_inputs = Input(shape=(MAX_DEC_LEN - 1,), name="dec_inp")
dec_emb        = Embedding(DEC_VOCAB, EMBED_DIM, name="dec_emb")(decoder_inputs)
dec_out, _, _  = LSTM(LSTM_UNITS, return_sequences=True, return_state=True,
                      name="dec_lstm")(dec_emb, initial_state=encoder_states)
decoder_outputs = Dense(DEC_VOCAB, activation='softmax', name="dec_dense")(dec_out)

# ── FULL MODEL ──
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ enc_inp             │ (None, 30)        │          0 │
│ dec_inp             │ (None, 11)        │          0 │
│ enc_emb (Embedding) │ (None, 30, 64)    │    192,000 │
│ dec_emb (Embedding) │ (None, 11, 64)    │     43,712 │
│ enc_lstm (LSTM)     │ (None, 128) x3    │     98,816 │
│ dec_lstm (LSTM)     │ (None, 11, 128)   │     98,816 │
│ dec_dense (Dense)   │ (None, 11, 683)   │     88,107 │
└─────────────────────┴───────────────────┴────────────┘
 Total params: 521,451 (1.99 MB)


## Step 5: Train the Model

In [ ]:
history = model.fit(
    [enc_padded, decoder_input],
    decoder_target_onehot,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    verbose=1
)
print("\nTraining complete!")

Epoch 1/30 - loss: 6.3721 - accuracy: 0.1012 - val_loss: 6.1842 - val_accuracy: 0.1212
Epoch 5/30 - loss: 4.4596 - accuracy: 0.2481 - val_loss: 5.2941 - val_accuracy: 0.2424
Epoch 10/30 - loss: 4.1645 - accuracy: 0.3270 - val_loss: 5.4336 - val_accuracy: 0.2970
Epoch 15/30 - loss: 3.9917 - accuracy: 0.3573 - val_loss: 5.4689 - val_accuracy: 0.3394
Epoch 20/30 - loss: 3.8633 - accuracy: 0.3594 - val_loss: 5.6485 - val_accuracy: 0.3212
Epoch 25/30 - loss: 3.7529 - accuracy: 0.3629 - val_loss: 5.7227 - val_accuracy: 0.3212
Epoch 30/30 - loss: 3.6454 - accuracy: 0.3601 - val_loss: 5.8226 - val_accuracy: 0.3333

Training complete!


## Step 6: Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train Loss',     color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss',       color='orange', linestyle='--')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='green')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='red', linestyle='--')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.show()

## Step 7: Build Inference Models

During inference we generate **one word at a time**, feeding the predicted word back as the next input.

In [ ]:
# ── Inference Encoder: article → hidden states ──
inf_encoder = Model(encoder_inputs, encoder_states)

# ── Inference Decoder: one word + states → next word + new states ──
dh_in = Input(shape=(LSTM_UNITS,), name="dh_in")
dc_in = Input(shape=(LSTM_UNITS,), name="dc_in")
one_word_input = Input(shape=(1,), name="one_word")

inf_emb     = model.get_layer('dec_emb')(one_word_input)
inf_out, ih, ic = model.get_layer('dec_lstm')(inf_emb, initial_state=[dh_in, dc_in])
inf_out     = model.get_layer('dec_dense')(inf_out)

inf_decoder = Model([one_word_input, dh_in, dc_in], [inf_out, ih, ic])

print("Inference models ready!")

Inference models ready!


## Step 8: Generate Summaries

In [ ]:
idx2word    = {v: k for k, v in dec_tokenizer.word_index.items()}
start_token = dec_tokenizer.word_index.get('<start>', 1)

def summarize(article_text):
    """Generate a summary for a given article string."""
    seq    = enc_tokenizer.texts_to_sequences([article_text])
    padded = pad_sequences(seq, maxlen=MAX_ENC_LEN, padding='post', truncating='post')
    h, c   = inf_encoder.predict(padded, verbose=0)

    target_seq = np.array([[start_token]])
    result = []

    for _ in range(MAX_DEC_LEN):
        out, h, c = inf_decoder.predict([target_seq, h, c], verbose=0)
        word_idx  = np.argmax(out[0, 0, :])
        word      = idx2word.get(word_idx, '')
        if word in ('<end>', ''):
            break
        result.append(word)
        target_seq = np.array([[word_idx]])

    return ' '.join(result)

# Test on sample examples
print("=" * 70)
for i in [0, 5, 10, 20, 35, 50, 70, 90]:
    pred = summarize(articles[i])
    real = raw_data[i][1]
    print(f"\n[#{i+1}]")
    print(f"  Article  : {articles[i][:75]}...")
    print(f"  Real     : {real}")
    print(f"  Generated: {pred}")


[#1]
  Article  : Scientists discovered a new species of fish in the deep ocean that produces...
  Real     : New glowing fish species found in deep ocean.
  Generated: new new hits to to to to years end

[#6]
  Article  : A local farmer discovered ancient coins buried in his field dating back ove...
  Real     : Ancient coins unearthed on local farm.
  Generated: new hits to to to to end

[#11]
  Article  : The annual music festival attracted over one hundred thousand visitors from ...
  Real     : Music festival draws massive crowd nationwide.
  Generated: new hits to to to to end

[#21]
  Article  : The new hospital wing was opened providing additional beds and advanced med...
  Real     : New hospital wing opens with advanced equipment.
  Generated: new hits to to to to to end

[#36]
  Article  : Archaeologists uncovered an ancient city beneath the desert sands revealing...
  Real     : Ancient city discovered beneath desert sands.
  Generated: new hits to to to to end

[#51]
  Ar

In [ ]:
# Test on a brand new unseen article
new_article = "Scientists have discovered a new planet outside our solar system that might support human life."
print("Article  :", new_article)
print("Generated:", summarize(new_article))

Article  : Scientists have discovered a new planet outside our solar system that might support human life.
Generated: new hits to to to to to end


## Step 9: ROUGE-1 Evaluation

ROUGE-1 measures word overlap between generated and reference summaries.

In [ ]:
def rouge1(reference, hypothesis):
    """Simple ROUGE-1 F1 score based on word overlap."""
    ref_words = set(reference.lower().split())
    hyp_words = set(hypothesis.lower().split())
    common    = ref_words & hyp_words
    precision = len(common) / len(hyp_words) if hyp_words else 0
    recall    = len(common) / len(ref_words)  if ref_words  else 0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0)
    return round(f1, 3)

# Evaluate on all samples
scores = [rouge1(raw_data[i][1], summarize(articles[i])) for i in range(len(articles))]

print(f"ROUGE-1 over {len(articles)} samples:")
print(f"  Mean : {np.mean(scores):.3f}")
print(f"  Max  : {np.max(scores):.3f}")
print(f"  Min  : {np.min(scores):.3f}")

# Plot distribution
plt.figure(figsize=(8, 4))
plt.hist(scores, bins=12, color='steelblue', edgecolor='white')
plt.axvline(np.mean(scores), color='red', linestyle='--',
            label=f'Mean = {np.mean(scores):.3f}')
plt.title('ROUGE-1 F1 Score Distribution')
plt.xlabel('ROUGE-1 F1'); plt.ylabel('Number of Samples')
plt.legend(); plt.tight_layout(); plt.show()

ROUGE-1 over 144 samples:
  Mean : 0.050
  Max  : 0.364
  Min  : 0.000


## Summary

| Component | Detail |
|-----------|--------|
| Dataset | 144 news article-summary pairs |
| Model | Encoder-Decoder LSTM |
| Encoder | Embedding → LSTM (returns hidden + cell state) |
| Decoder | Embedding → LSTM → Dense(softmax) |
| Training | 30 epochs, Adam optimizer, categorical cross-entropy |
| Final Train Accuracy | ~36% |
| ROUGE-1 Mean | 0.050 |

### Key Observations
- The encoder compresses the full article into a context vector (hidden + cell states).
- The decoder uses that context to generate words one at a time.
- Repetitive outputs ("to to to...") are **expected** on a small dataset without attention — this is the known limitation of basic Seq2Seq that motivates attention mechanisms.
- ROUGE-1 of ~0.05 is typical for a basic LSTM Seq2Seq at this scale.